In [ ]:
# importing the required packages
import czifile # to import a .czi file
# from PIL import Image # to convert .czi file to a .tif file
import skimage # general package for manipulating imaging data
from pathlib import Path # for file path 
import numpy as np
import matplotlib.pyplot as plt
from microfilm.microplot import microshow # for viewing multichannel image data
from skimage.transform import rotate # to rotate the image as a control
from skimage.restoration import rolling_ball # for image processing
from skimage.filters import gaussian # for image processing
from skimage.feature import peak_local_max # for local max detection
import sys
import os

import optuna
import pandas as pd
import re

# with this peice of code, it will recognize the custom modules
project_root = "/Users/cgeyskens/Documents/code/phd/image-analysis/synapse-counting"
sys.path.append(project_root)

# custom modules
from synapse_counting import metadata, preprocessing, calc_synaptic_metrics, calc_synaptic_coloc

In [ ]:
def local_peak_detection(presynapse_preprocessed, postsynapse_preprocessed, presynapse_distance, postsynapse_distance, presynapse_threshold, postsynapse_threshold, plot_coord = False):
    """
    Detects the local intensity peaks of the presynapse and the postsynapse channel.
    
        Args:
            presynapse_preprocessed (np.array): processed image of presynapse, which is background substracted and has a gaussian blur
            postsynapse_preprocessed (np.array): processed image of postsynapse, which is background substracted and has a gaussian blur
            presynapse_distance (int): minimun distance between presynapse local peak maxima, usually 1
            postsynapse_distance (int): mimimun distance between postsynapse local peak maxima, usually 1
            presynapse_threshold (float): thresholding of the presynapse image for local peak detection
            postsynapse_threshold (float): thresholding of the postsynapse image for local peak detection
            plot_coordinates (bool): option to visualize the images
    
        Returns:
            presynapse_coord (np.array): coordinates of presynapse local peak maxima
            postsynapse_coord (np.array): coordinates of postsynapse local peak maxima
            postsynapse_coord_rot (np.array): coordinates of postsynapse local peak maxima, but rotated
            
            plot of vglut1, psd95, psd95_rot images with the local peak maxima overlaid
    """
    
    # Thresholding the image for local peak maximum detection
    presynapse_coord = peak_local_max(presynapse_preprocessed, min_distance = presynapse_distance, threshold_abs = presynapse_threshold)
    postsynapse_coord = peak_local_max(postsynapse_preprocessed, min_distance = postsynapse_distance, threshold_abs = postsynapse_threshold)

    # Rotating an image (psd95) as a control
    postsynapse_rot = rotate(postsynapse_preprocessed, 90)
    postsynapse_rot_coord = peak_local_max(postsynapse_rot, min_distance = postsynapse_distance, threshold_abs = postsynapse_threshold)
    
    if plot_coord == True:
        # Showing the local peaks with coordinates together with the images
        fig, axs = plt.subplots(2, 2, figsize=(30, 30))

        axs[0,0].imshow(presynapse_preprocessed, cmap='gray')
        # axs[0,0].plot(vglut1_coord[:, 1], vglut1_coord[:, 0], 'c.')
        axs[0,0].set_title('vglut1_pre')

        axs[0,1].imshow(postsynapse_preprocessed, cmap='gray')
        # axs[0,1].plot(psd95_coord[:, 1], psd95_coord[:, 0], 'm.')
        axs[0,1].set_title('psd95_pre')

        axs[1,0].imshow(presynapse_preprocessed, cmap='gray')
        axs[1,0].plot(presynapse_coord[:, 1], presynapse_coord[:, 0], 'c.')
        axs[1,0].set_title('vglut1_pre')

        axs[1,1].imshow(postsynapse_preprocessed, cmap='gray')
        axs[1,1].plot(postsynapse_coord[:, 1], postsynapse_coord[:, 0], 'm.')
        axs[1,1].set_title('psd95_pre')
    else:
        pass
    
    return presynapse_coord, postsynapse_coord, postsynapse_rot_coord

In [ ]:
from scipy.spatial.distance import cdist

def count_coloc_spots(presynapse_coordinates, postsynapse_coordinates, pixel_size_um, max_distance_um):
    """
    Counts the number of colocalized pre and postsynaptic spots based on the coordinates array of the local maxima detection.
    
    Args:
        presynapse_coordinates (np.array): coordinates of the local peak maxima in the vlgut1 channel
        postsynapse_coordinates (np.array): coordinates of the local peak maxima in the psd95 channel
        pixel_size_um (float): size of a pixel in um, based on image settings
        max_distance_um (float): value of the maximum colocalization distance between the spots of each channel in um
    
    Returns:
        colocalized spot count (int): the number of colocalized pre and postsynaptic spots.
    """
    
    # calculate the max distance in pixels with the pixel_size_um and max_distance_um
    max_distance_px = max_distance_um/pixel_size_um
    
    # calculate pairwise distances between spots in vlgut1 and psd95 channel
    distances_pre_to_post = cdist(presynapse_coordinates, postsynapse_coordinates)
    distances_post_to_pre = cdist(postsynapse_coordinates, presynapse_coordinates)

    # find unique colocalized spots
    colocalized_spots = set()

    # tterate over distances from presynapse_coordinates to postsynapse_coordinates
    for i in range(len(presynapse_coordinates)):
        # Check if the current spot in vlgut1 has nearby spots in psd95
        colocalized_indices = [j for j, distance in enumerate(distances_pre_to_post[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((i, j))

    # tterate over distances from postsynapse_coordinates to presynapse_coordinates
    for i in range(len(postsynapse_coordinates)):
        # check if the current spot in Channel 2 has nearby spots in Channel 1
        colocalized_indices = [j for j, distance in enumerate(distances_post_to_pre[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((j, i))

    # get the count of unique colocalized spots
    colocalized_spot_count = len(colocalized_spots)
    
    return colocalized_spot_count

In [ ]:
# input folder
input_folder = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images"

# get a list of files in that input_folder
file_list = os.listdir(input_folder)
print(file_list)

protein_and_synaptic_marker = "VCAM1_LacZ_VGLUT1_PSD95"

In [ ]:
import dask
from dask import delayed, compute
import dask.multiprocessing

In [ ]:
def load_and_preprocess(file_path, presynapse_channel, postsynapse_channel, preprocess_params):
    # extract metadata
    pixel_size_um, _ , _ = metadata.extract_metadata(file_path)
    # extract channels
    pre, post = preprocessing.extract_and_split(file_path, presynapse_channel = presynapse_channel, postsynapse_channel = postsynapse_channel)
    # preprocessing
    p = preprocessing.ImagePreprocessing(
        include_rolling_ball=preprocess_params['include_rolling_ball'], radius=preprocess_params['radius'],
        include_blur=preprocess_params['include_blur'], sigma=preprocess_params['sigma'], preserve_range=True,
        include_clahe=preprocess_params['include_clahe'],
        include_tophat=preprocess_params['include_tophat'], element_size=preprocess_params['element_size']
    )
    pre_processed, post_processed = p.preprocess(pre, post)
    return pre_processed, post_processed, pixel_size_um

In [ ]:
def objective(trial, presynapse_preprocessed, postsynapse_preprocessed, pixel_size_um, param_ranges):
    
    pre_distance = trial.suggest_int("pre_distance", *param_ranges['pre_distance'])
    post_distance = trial.suggest_int("post_distance", *param_ranges['post_distance'])
    pre_threshold = trial.suggest_float("pre_threshold", *param_ranges['pre_threshold'])
    post_threshold = trial.suggest_float("post_threshold", *param_ranges['post_threshold'])
    max_distance_um = trial.suggest_float("max_distance_um", *param_ranges['max_distance_um'])

    # probing the first function to get the parameters as input for the second function
    pre_coord, post_coord, post_rot_coord = local_peak_detection(presynapse_preprocessed = presynapse_preprocessed,
                                                                 postsynapse_preprocessed = postsynapse_preprocessed, 
                                                                 presynapse_distance = pre_distance, 
                                                                 postsynapse_distance = post_distance, 
                                                                 presynapse_threshold = pre_threshold, 
                                                                 postsynapse_threshold = post_threshold)
                                                                 
    # probing the second function where psd95 is not rotated, the actual condition
    colocalized_spot_count = count_coloc_spots(pre_coord, post_coord, pixel_size_um, max_distance_um)
    # probing the second function where psd95 is rotated, the internal control condition
    colocalized_spot_count_rot = count_coloc_spots(pre_coord, post_rot_coord, pixel_size_um, max_distance_um)
    
    # getting the scale differences between the spot count of the normal situation and psd95 rotated
    scaled_difference = (colocalized_spot_count - colocalized_spot_count_rot) / max(colocalized_spot_count, 1)
    
    return scaled_difference

In [ ]:
# main optimization process for each hippocampal_layer, without dask
def optimize_parameters_for_hippocampal_layer(file_list, input_folder, hippocampal_layers, preprocess_params_by_hippocampal_layer, param_ranges, nr_of_trials):
    
    best_params_by_hippocampal_layer = {}
    results_dfs = []
    all_trials_data = []
    
    for hippocampal_layer in hippocampal_layers:
        images = [f for f in file_list if hippocampal_layer in f and "LacZ-gRNA" in f] # only taking control (LacZ-images) images for setting the parameters
    
        preprocess_params = preprocess_params_by_hippocampal_layer[hippocampal_layer]
        
        # extract metadata and preprocess images
        image_results = []
        for file_name in images:
            file_path = os.path.join(input_folder, file_name)
            pre_1, post_1, pixel_size_um = load_and_preprocess(file_path, presynapse_channel = 0, postsynapse_channel = 1, preprocess_params = preprocess_params)
            if pre_1 is not None and post_1 is not None:
                image_results.append((pre_1, post_1, pixel_size_um, file_path))
        
        # optuna optimization
        def hippocampal_layer_objective(trial):
            detailed_trial_results = []
            for pre_1, post_1, pixel_size_um, file_path in image_results:
                score = objective(trial, pre_1, post_1, pixel_size_um, param_ranges[hippocampal_layer])
                detailed_trial_results.append({
                    'hippocampal_layer': hippocampal_layer,
                    'trial': trial.number,
                    'image': file_path,
                    'params': trial.params,
                    'score': score
                })
            mean_score = np.mean([result['score'] for result in detailed_trial_results])
            return mean_score, detailed_trial_results
        
        # create the study
        study = optuna.create_study(study_name=hippocampal_layer, direction="maximize", sampler=optuna.samplers.TPESampler())
        detailed_trials_data = []

        # define the optuna objective, such that the mean score of all the images in a certain hippocampal layer is optimized
        def optuna_objective(trial):
            mean_score, detailed_trial_results = hippocampal_layer_objective(trial)
            detailed_trials_data.extend(detailed_trial_results)
            return mean_score
        
        # do the optimization
        study.optimize(optuna_objective, n_trials = nr_of_trials)
    
        # store best parameters for the hippocampal_layer with the corresponding score
        best_trial_full = study.best_trial
        best_params_by_hippocampal_layer[hippocampal_layer] = {
            "best_trial": best_trial_full.number,
            "best_params": study.best_params,
            "best_score": study.best_value
            }
        
        # create dataframe for detailed trial results and the best params
        trials_df = pd.DataFrame(detailed_trials_data)
        results_dfs.append(trials_df)

        # getting the data per trial
        all_trials_data.extend([{'hippocampal_layer': hippocampal_layer} | trial for trial in study.trials_dataframe().to_dict('records')])

    # create dataframe for best parameters by hippocampal_layer
    best_params_by_hippocampal_layer_df = pd.DataFrame.from_dict(best_params_by_hippocampal_layer, orient = "index").reset_index()
    best_params_by_hippocampal_layer_df = pd.concat([best_params_by_hippocampal_layer_df.drop(['best_params'], axis=1), 
                                pd.json_normalize(best_params_by_hippocampal_layer_df['best_params'])], axis=1) # from json to columns

    # combine dataframes for all hippocampal_layers
    final_df = pd.concat(results_dfs, ignore_index=True)

    # combine the trial data
    final_df_optuna = pd.DataFrame(all_trials_data)

    return best_params_by_hippocampal_layer_df, final_df, final_df_optuna


In [ ]:
import os
import dask
from dask import delayed, compute
import numpy as np
import pandas as pd
import optuna

def optimize_parameters_for_hippocampal_layer(file_list, 
                                              input_folder, 
                                              hippocampal_layers, 
                                              preprocess_params_by_hippocampal_layer, 
                                              param_ranges, 
                                              nr_of_trials):
    """
    This function will optimize the parameters for colocalization based on the objective function.

    It uses Tree-structured Parzen Estimator (TPE) from the optuna library to optimize the parameters.
    For each batch of presynapse and postsynapse images per hippocampal layer, it will try to optimize the 
    parameters such that the scaled difference between actual and rotated presynapse/postsynapses images is 
    maximized. It also uses dask such that it parallel process each batch of images per hippocampal layer.
    
    Args:
        file_list: a list of files (images) that will be processed.
        input_folder: the folder where the images are stored.
        hippocampal_layers: a list of hippocampal layers of which the images needs to be optimized.
        preprocess_params_by_hippocampal_layer: a list of unique handcrafted preprocessing parameters for each hippocampal layer
        param_ranges: a list of unique handcrafted parameters ranges for each hippocampal layer
        nr_of_trials: the total nr of trials to optimize the parameters

    Returns:
        best_params_by_hippocampal_layer_df: dataframe containing the best parameters per hippocampal layer
        final_df_optuna: dataframe containing the info per trial.
        final_df: dataframe containing all the optimization information, including the the scores for each image combination
    """
    
    @dask.delayed
    def process_hippocampal_layer(hippocampal_layer):
        images = [f for f in file_list if hippocampal_layer in f and "LacZ-gRNA" in f] # only taking control (LacZ-images) images for setting the parameters
    
        preprocess_params = preprocess_params_by_hippocampal_layer[hippocampal_layer]
        
        # extract metadata and preprocess images
        image_results = []
        for file_name in images:
            file_path = os.path.join(input_folder, file_name)
            pre_1, post_1, pixel_size_um = load_and_preprocess(file_path, presynapse_channel=0, postsynapse_channel=1, preprocess_params=preprocess_params)
            if pre_1 is not None and post_1 is not None:
                image_results.append((pre_1, post_1, pixel_size_um, file_path))
        
        # optuna optimization
        def hippocampal_layer_objective(trial):
            detailed_trial_results = []
            for pre_1, post_1, pixel_size_um, file_path in image_results:
                score = objective(trial, pre_1, post_1, pixel_size_um, param_ranges[hippocampal_layer])
                detailed_trial_results.append({
                    'hippocampal_layer': hippocampal_layer,
                    'trial': trial.number,
                    'image': file_path,
                    'params': trial.params,
                    'score': score
                })
            mean_score = np.mean([result['score'] for result in detailed_trial_results])
            return mean_score, detailed_trial_results
        
        # create the study
        study = optuna.create_study(study_name=hippocampal_layer, direction="maximize", sampler=optuna.samplers.TPESampler())
        detailed_trials_data = []

        # define the optuna objective, such that the mean score of all the images in a certain hippocampal layer is optimized
        def optuna_objective(trial):
            mean_score, detailed_trial_results = hippocampal_layer_objective(trial)
            detailed_trials_data.extend(detailed_trial_results)
            return mean_score
        
        # do the optimization
        study.optimize(optuna_objective, n_trials=nr_of_trials, show_progress_bar = True)
    
        # store best parameters for the hippocampal_layer with the corresponding score
        best_trial_full = study.best_trial
        best_params = {
            "best_trial": best_trial_full.number,
            "best_params": study.best_params,
            "best_score": study.best_value
        }
        
        # create dataframe for detailed trial results and the best params
        trials_df = pd.DataFrame(detailed_trials_data)
        
        # getting the data per trial
        trials_data = [{'hippocampal_layer': hippocampal_layer} | trial for trial in study.trials_dataframe().to_dict('records')]

        return hippocampal_layer, best_params, trials_df, trials_data

    # Collect the delayed tasks
    delayed_tasks = [process_hippocampal_layer(hippocampal_layer) for hippocampal_layer in hippocampal_layers]

    # Compute the results in parallel
    results = dask.compute(*delayed_tasks)

    best_params_by_hippocampal_layer = {}
    results_dfs = []
    all_trials_data = []

    for hippocampal_layer, best_params, trials_df, trials_data in results:
        best_params_by_hippocampal_layer[hippocampal_layer] = best_params
        results_dfs.append(trials_df)
        all_trials_data.extend(trials_data)

    # create dataframe for best parameters by hippocampal_layer
    best_params_by_hippocampal_layer_df = pd.DataFrame.from_dict(best_params_by_hippocampal_layer, orient="index").reset_index()
    best_params_by_hippocampal_layer_df = pd.concat([best_params_by_hippocampal_layer_df.drop(['best_params'], axis=1), 
                                          pd.json_normalize(best_params_by_hippocampal_layer_df['best_params'])], axis=1) # from json to columns

    # combine dataframes for all hippocampal_layers
    final_df = pd.concat(results_dfs, ignore_index=True)

    # combine the trial data
    final_df_optuna = pd.DataFrame(all_trials_data)

    return best_params_by_hippocampal_layer_df, final_df_optuna, final_df


In [ ]:
# Example usage
input_folder = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images"
file_list = os.listdir(input_folder)

hippocampal_layers = ["CA1_SR", "CA1_SO", "CA1_SLM", "CA3_SO", "CA3_SL", "CA3_SR", "DG_Hilus", "DG_ML"]  # Example hippocampal_layers

preprocess_params_by_hippocampal_layer = {
    'CA1_SO': {
        'include_rolling_ball': True, 'radius': 5,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 10
    },
    'CA1_SR': {
        'include_rolling_ball': True, 'radius': 10,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 10
    },
    'CA1_SLM': {
        'include_rolling_ball': True, 'radius': 5,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 5
    },
    'CA3_SO': {
        'include_rolling_ball': True, 'radius': 5,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 10
    },
    'CA3_SL': {
        'include_rolling_ball': True, 'radius': 15,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 20
    },
    'CA3_SR': {
        'include_rolling_ball': True, 'radius': 10,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 10
    },
    'DG_Hilus': {
        'include_rolling_ball': True, 'radius': 15,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 20
    },
    'DG_ML': {
        'include_rolling_ball': True, 'radius': 5,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 15
    },
}

param_ranges = {
    'CA1_SO': {
        'pre_distance': (1, 5),
        'post_distance': (1,5),
        'pre_threshold': (500, 900),
        'post_threshold': (100, 150),
        'max_distance_um': (0.01, 1)
    },
    'CA1_SR': {
        'pre_distance': (1, 5),
        'post_distance': (1,5),
        'pre_threshold': (300, 500),
        'post_threshold': (100, 200),
        'max_distance_um': (0.01, 1)
    },
    'CA1_SLM': {
        'pre_distance': (1, 5),
        'post_distance': (1,5),
        'pre_threshold': (150, 250),
        'post_threshold': (50, 80),
        'max_distance_um': (0.01, 1)
    },
    'CA3_SO': {
        'pre_distance': (1, 5),
        'post_distance': (1,5),
        'pre_threshold': (500, 1200),
        'post_threshold': (100, 160),
        'max_distance_um': (0.01, 1)
    },
    'CA3_SL': {
        'pre_distance': (1, 10),
        'post_distance': (1, 10),
        'pre_threshold': (1000, 2000),
        'post_threshold': (250, 400),
        'max_distance_um': (0.01, 1)
    },
    'CA3_SR': {
        'pre_distance': (1, 5),
        'post_distance': (1, 5),
        'pre_threshold': (500, 1000),
        'post_threshold': (100, 200),
        'max_distance_um': (0.01, 1)
    },
    'DG_Hilus': {
        'pre_distance': (1, 10),
        'post_distance': (1, 10),
        'pre_threshold': (1000, 3000),
        'post_threshold': (300, 500),
        'max_distance_um': (0.01, 1)
    },
    'DG_ML': {
        'pre_distance': (1, 10),
        'post_distance': (1, 10),
        'pre_threshold': (500, 1000),
        'post_threshold': (100, 200),
        'max_distance_um': (0.01, 1)
    }
}

nr_of_trials = 5

best_params_by_hippocampal_layer_df, final_df, final_df_optuna = optimize_parameters_for_hippocampal_layer(file_list, input_folder, hippocampal_layers, preprocess_params_by_hippocampal_layer, param_ranges, nr_of_trials)

In [ ]:
best_params_by_hippocampal_layer_df

In [ ]:
# PROBLEM:
# Also, there must be a nicer way for inserting the preprocessing parameters and the optimized parameters

In [ ]:
synaptic_marker = "VGLUT1_PSD95"
name_segments = "0 1 2 3 6 8 9"
# Split the string by spaces
string_list = name_segments.split(" ")
# Convert each string element to an integer
name_segments = [int(element) for element in string_list]

presynapse_channel = "0"
postsynapse_channel = "1"
input_folder = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images"
output_folder = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_output_data"

# get a list of files in that input_folder
file_list = os.listdir(input_folder)
print(file_list)

protein_and_synaptic_marker = "VCAM1_LacZ_VGLUT1_PSD95"

In [ ]:
# I need: presynapse_distance, postsynapse_distance, presynapse_threshold, postsynapse_threshold, max_distance

In [ ]:
best_params_by_hippocampal_layer_df


In [ ]:
best_params_by_hippocampal_layer_df.set_index('index', inplace=True)

In [ ]:
best_params_by_hippocampal_layer_df

In [ ]:
synaptic_marker

In [ ]:
preprocessing_params = {
    "VGLUT1_PSD95": {
        "CA3_SL": {"radius": 15, "element_size": 20, "blur_sigma": 2},
        "DG_Hilus": {"radius": 15, "element_size": 20, "blur_sigma": 2},
        "CA1_SO": {"radius": 5, "element_size": 10, "blur_sigma": 2},
        "CA3_SO": {"radius": 5, "element_size": 10, "blur_sigma": 2},
        "CA1_SR": {"radius": 10, "element_size": 10, "blur_sigma": 2},
        "CA3_SR": {"radius": 10, "element_size": 10, "blur_sigma": 2},
        "CA1_SLM": {"radius": 5, "element_size": 5, "blur_sigma": 2},
        "DG_ML": {"radius": 5, "element_size": 15, "blur_sigma": 2}
    },
    "VGLUT2_PSD95": {
        "Cortex_L4": {"radius": 10, "element_size": 10, "blur_sigma": 2},
        "CA2_SP": {"radius": 15, "element_size": 10, "blur_sigma": 2},
        "DG_GC": {"radius": 15, "element_size": 10, "blur_sigma": 2},
        "Subiculum_SP": {"radius": 15, "element_size": 10, "blur_sigma": 2}
    }
}


In [ ]:
# Get the parameters based on synaptic_marker and result
result = "CA1_SLM"

params = preprocessing_params.get(synaptic_marker, {}).get(result, {})

radius = params.get("radius")
element_size = params.get("element_size")
blur_sigma = params.get("blur_sigma")

params

In [ ]:
best_params_by_hippocampal_layer_df.at["CA1_SR", "pre_distance"]

In [ ]:
# the operation
@dask.delayed
def local_peak_colocalizaion(filename, name_segments, best_params_df):
    """
    Measure colocalized local peaks with optimized parameters. Decorated by dask delayed for parrellel processing. 
    """    
    if not filename.endswith(".czi"):
        return None
    
    file_path = os.path.join(input_folder, filename)
    parts = filename.split('_')[-2:]
    result = "_".join(parts)
    layer = result[:-4]
    print("layer:", layer)
    # set the index of the best_params_df
    # best_params_df.set_index('index', inplace=True)

    # getting only the params from a distinct layer
    params = preprocessing_params.get(synaptic_marker, {}).get(layer, {})
    
    # for each synaptic_marker combination and layer, I handcrafted the preprocessing parameters, the colocalization params are optimized.
    if synaptic_marker == "VGLUT1_PSD95":
        if layer == "CA3_SL":
            print("Yes, CA3_SL")
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["CA3_SL", "pre_distance"]
            postsynapse_distance = best_params_df.at["CA3_SL", "post_distance"]
            presynapse_threshold = best_params_df.at["CA3_SL", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["CA3_SL", "post_threshold"]
            max_distance = best_params_df.at["CA3_SL", "max_distance_um"]
        elif layer == "DG_Hilus":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["DG_Hilus", "pre_distance"]
            postsynapse_distance = best_params_df.at["DG_Hilus", "post_distance"]
            presynapse_threshold = best_params_df.at["DG_Hilus", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["DG_Hilus", "post_threshold"]
            max_distance = best_params_df.at["DG_Hilus", "max_distance_um"]
        elif layer == "CA1_SO":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["CA1_SO", "pre_distance"]
            postsynapse_distance = best_params_df.at["CA1_SO", "post_distance"]
            presynapse_threshold = best_params_df.at["CA1_SO", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["CA1_SO", "post_threshold"]
            max_distance = best_params_df.at["CA1_SO", "max_distance_um"]
        elif layer == "CA3_SO":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["CA3_SO", "pre_distance"]
            postsynapse_distance = best_params_df.at["CA3_SO", "post_distance"]
            presynapse_threshold = best_params_df.at["CA3_SO", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["CA3_SO", "post_threshold"]
            max_distance = best_params_df.at["CA3_SO", "max_distance_um"]
        elif layer == "CA1_SR":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["CA1_SR", "pre_distance"]
            postsynapse_distance = best_params_df.at["CA1_SR", "post_distance"]
            presynapse_threshold = best_params_df.at["CA1_SR", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["CA1_SR", "post_threshold"]
            max_distance = best_params_df.at["CA1_SR", "max_distance_um"]
        elif layer == "CA3_SR":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["CA3_SR", "pre_distance"]
            postsynapse_distance = best_params_df.at["CA3_SR", "post_distance"]
            presynapse_threshold = best_params_df.at["CA3_SR", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["CA3_SR", "post_threshold"]
            max_distance = best_params_df.at["CA3_SR", "max_distance_um"]
        elif layer == "CA1_SLM":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["CA1_SLM", "pre_distance"]
            postsynapse_distance = best_params_df.at["CA1_SLM", "post_distance"]
            presynapse_threshold = best_params_df.at["CA1_SLM", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["CA1_SLM", "post_threshold"]
            max_distance = best_params_df.at["CA1_SLM", "max_distance_um"]
        elif layer == "DG_ML":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["CA1_SLM", "pre_distance"]
            postsynapse_distance = best_params_df.at["CA1_SLM", "post_distance"]
            presynapse_threshold = best_params_df.at["CA1_SLM", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["CA1_SLM", "post_threshold"]
            max_distance = best_params_df.at["CA1_SLM", "max_distance_um"]
    elif synaptic_marker == "VGLUT2_PSD95":
        if layer == "Cortex_L4":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["Cortex_L4", "pre_distance"]
            postsynapse_distance = best_params_df.at["Cortex_L4", "post_distance"]
            presynapse_threshold = best_params_df.at["Cortex_L4", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["Cortex_L4", "post_threshold"]
            max_distance = best_params_df.at["Cortex_L4", "max_distance_um"]
        elif layer == "CA2_SP":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["CA2_SP", "pre_distance"]
            postsynapse_distance = best_params_df.at["CA2_SP", "post_distance"]
            presynapse_threshold = best_params_df.at["CA2_SP", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["CA2_SP", "post_threshold"]
            max_distance = best_params_df.at["CA2_SP", "max_distance_um"]
        elif layer == "DG_GC":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["DG_GC", "pre_distance"]
            postsynapse_distance = best_params_df.at["DG_GC", "post_distance"]
            presynapse_threshold = best_params_df.at["DG_GC", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["DG_GC", "post_threshold"]
            max_distance = best_params_df.at["DG_GC", "max_distance_um"]
        elif layer == "Subiculum_SP":
            radius = params.get("radius")
            element_size = params.get("element_size")
            blur_sigma = params.get("blur_sigma")
            presynapse_distance = best_params_df.at["Subiculum_SP", "pre_distance"]
            postsynapse_distance = best_params_df.at["Subiculum_SP", "post_distance"]
            presynapse_threshold = best_params_df.at["Subiculum_SP", "pre_threshold"]
            postsynapse_threshold = best_params_df.at["Subiculum_SP", "post_threshold"]
            max_distance = best_params_df.at["Subiculum_SP", "max_distance_um"]

    # extract metadata for pixel_size parameter in overlap_um2_coloc
    pixel_size_um, _ , _ = metadata.extract_metadata(file_path)

    # extracting and splitting channels
    pre, post = preprocessing.extract_and_split(file_path, 
                                                presynapse_channel = 0, 
                                                postsynapse_channel = 1
                                                )
    # preprocessing
    p = preprocessing.ImagePreprocessing(
        include_rolling_ball=True, radius=radius,
        include_blur=True, sigma = blur_sigma, preserve_range = True,
        include_clahe=False,
        include_tophat=True, element_size = element_size
        )
    pre_1, post_1 = p.preprocess(pre, post)
    # calculating the coordinates
    presynapse_coord, postsynapse_coord, postsynapse_rot_coord = calc_synaptic_coloc.local_peak_detection(
        presynapse_preprocessed=pre_1,
        postsynapse_preprocessed=post_1,
        presynapse_distance = presynapse_distance,
        postsynapse_distance= postsynapse_distance,
        presynapse_threshold= presynapse_threshold,
        postsynapse_threshold= postsynapse_threshold,
        plot_coord = False
        )                                                                                                  
    # actual colocalized synapses
    colocalized_spots = calc_synaptic_coloc.count_coloc_spots(
        presynapse_coordinates = presynapse_coord,
        postsynapse_coordinates = postsynapse_coord,
        pixel_size_um = pixel_size_um,
        max_distance_um = max_distance
        )
    # rotated colocalized synapses as control
    colocalized_spots_rot = calc_synaptic_coloc.count_coloc_spots(
        presynapse_coordinates = presynapse_coord,
        postsynapse_coordinates = postsynapse_rot_coord,
        pixel_size_um = pixel_size_um,
        max_distance_um = max_distance
        )
    # getting the right filename
    img_filename = metadata.image_filename(filename, name_segments)
    
    return {
        "img_filename": img_filename,
        "local_peak_colocalized_spots": colocalized_spots,
        "local_peak_colocalized_spots_rot": colocalized_spots_rot,
    }


In [ ]:
# get a list of files in that input_folder
file_list = os.listdir(input_folder)

# compute the results using a dask delayed object
delayed_results = [local_peak_colocalizaion(file_name, name_segments, best_params_df=best_params_by_hippocampal_layer_df) for file_name in file_list]
results = dask.compute(*delayed_results)

# reading out the results into a csv
df = pd.DataFrame(results)
output_csv_path = os.path.join(output_folder, "local_peak_coloc.csv")
df.to_csv(output_csv_path, encoding = "utf-8")

In [ ]:
# the operation
@dask.delayed
def local_peak_colocalizaion(filename, input_folder, preprocessing_params, synaptic_marker, optimzed_coloc_params_df, name_segments):
    """
    Measure colocalized local peaks with optimized parameters. Decorated by dask delayed for parrellel processing. 

    Args:
        filename: filename of the image
        inputfolder: the inputfolder with all the images
        preprocessing_params: dictionary with all the preprocessing params that were handcrafted
        synaptic_marker: combination of synaptic markers used
        optimzed_coloc_params_df: dataframe of the optimized params for colocalization
        name_segments: the import segments of the filename

    Returns:
        A dictionary containing the filename, the local peak colocalized spot and the 
        local peak colocalized spots when rotated.
    """    
    if not filename.endswith(".czi"):
        return None
    
    file_path = os.path.join(input_folder, filename)
    parts = filename.split('_')[-2:]
    result = "_".join(parts)
    layer = result[:-4]
    # set the index of the best_params_df
    # best_params_df.set_index('index', inplace=True)

    # getting only the params from a distinct layer
    params = preprocessing_params.get(synaptic_marker, {}).get(layer, {})

    # ectracting the best params for the handcrafted preprocessing params and the optimized colocalization params
    def extrac_all_params(layer):
        return {
            "radius": params.get("radius"),
            "element_size": params.get("element_size"),
            "blur_sigma":  params.get("blur_sigma"),
            "presynapse_distance": optimzed_coloc_params_df.at[layer, "pre_distance"],
            "postsynapse_distance": optimzed_coloc_params_df.at[layer, "post_distance"],
            "presynapse_threshold": optimzed_coloc_params_df.at[layer, "pre_threshold"],
            "postsynapse_threshold": optimzed_coloc_params_df.at[layer, "post_threshold"],
            "max_distance": optimzed_coloc_params_df.at[layer, "max_distance_um"]
            }
    all_param_values = extrac_all_params(layer)

    # extract metadata for pixel_size parameter in overlap_um2_coloc
    pixel_size_um, _ , _ = metadata.extract_metadata(file_path)

    # extracting and splitting channels
    pre, post = preprocessing.extract_and_split(file_path, 
                                                presynapse_channel = 0, 
                                                postsynapse_channel = 1
                                                )
    # preprocessing
    p = preprocessing.ImagePreprocessing(
        include_rolling_ball=True, radius=all_param_values["radius"],
        include_blur=True, sigma = all_param_values["blur_sigma"], preserve_range = True,
        include_clahe=False,
        include_tophat=True, element_size = all_param_values["element_size"]
        )
    pre_1, post_1 = p.preprocess(pre, post)
    # calculating the coordinates
    presynapse_coord, postsynapse_coord, postsynapse_rot_coord = calc_synaptic_coloc.local_peak_detection(
        presynapse_preprocessed=pre_1,
        postsynapse_preprocessed=post_1,
        presynapse_distance = all_param_values["presynapse_distance"],
        postsynapse_distance= all_param_values["postsynapse_distance"],
        presynapse_threshold= all_param_values["presynapse_threshold"],
        postsynapse_threshold= all_param_values["postsynapse_threshold"],
        plot_coord = False
        )                                                                                                  
    # actual colocalized synapses
    colocalized_spots = calc_synaptic_coloc.count_coloc_spots(
        presynapse_coordinates = presynapse_coord,
        postsynapse_coordinates = postsynapse_coord,
        pixel_size_um = pixel_size_um,
        max_distance_um = all_param_values["max_distance"]
        )
    # rotated colocalized synapses as control
    colocalized_spots_rot = calc_synaptic_coloc.count_coloc_spots(
        presynapse_coordinates = presynapse_coord,
        postsynapse_coordinates = postsynapse_rot_coord,
        pixel_size_um = pixel_size_um,
        max_distance_um = all_param_values["max_distance"]
        )
    # getting the right filename
    img_filename = metadata.image_filename(filename, name_segments)
    
    return {
        "img_filename": img_filename,
        "local_peak_colocalized_spots": colocalized_spots,
        "local_peak_colocalized_spots_rot": colocalized_spots_rot,
    }


In [ ]:
# get a list of files in that input_folder
file_list = os.listdir(input_folder)

# compute the results using a dask delayed object
delayed_results = [local_peak_colocalizaion(filename = file_name, 
                                            input_folder = input_folder,
                                            preprocessing_params = preprocessing_params,
                                            synaptic_marker = synaptic_marker,
                                            optimzed_coloc_params_df = best_params_by_hippocampal_layer_df,
                                            name_segments = name_segments) for file_name in file_list]
results = dask.compute(*delayed_results)

# reading out the results into a csv
df = pd.DataFrame(results)
output_csv_path = os.path.join(output_folder, "local_peak_coloc.csv")
df.to_csv(output_csv_path, encoding = "utf-8")

In [ ]:
import os
import pandas as pd
import dask
from dask import delayed

@dask.delayed
def local_peak_colocalizaion(filename, name_segments, best_params_df, input_folder, preprocessing_params, synaptic_marker):
    """
    Measure colocalized local peaks with optimized parameters. Decorated by dask delayed for parallel processing. 
    """    
    if not filename.endswith(".czi"):
        return None
    
    file_path = os.path.join(input_folder, filename)
    parts = filename.split('_')[-2:]
    result = "_".join(parts)
    layer = result[:-4]
    print(layer)
    
    # Check if the layer exists in the DataFrame
    if layer not in best_params_df.index:
        print(f"Layer {layer} not found in best_params_df, skipping file {filename}")
        return None

    # getting only the params a distinct layer
    params = preprocessing_params.get(synaptic_marker, {}).get(layer, {})
    
    # Extract parameters based on the synaptic marker and layer
    def get_params(layer, params):
        radius = params.get("radius")
        element_size = params.get("element_size")
        blur_sigma = params.get("blur_sigma")
        presynapse_distance = best_params_df.at[layer, "pre_distance"]
        postsynapse_distance = best_params_df.at[layer, "post_distance"]
        presynapse_threshold = best_params_df.at[layer, "pre_threshold"]
        postsynapse_threshold = best_params_df.at[layer, "post_threshold"]
        max_distance = best_params_df.at[layer, "max_distance_um"]
        return radius, element_size, blur_sigma, presynapse_distance, postsynapse_distance, presynapse_threshold, postsynapse_threshold, max_distance

    try:
        radius, element_size, blur_sigma, presynapse_distance, postsynapse_distance, presynapse_threshold, postsynapse_threshold, max_distance = get_params(layer, params)
    except KeyError as e:
        print(f"Missing parameter for layer {layer}: {e}")
        return None
    
    print(radius, element_size, blur_sigma, presynapse_distance, postsynapse_distance, presynapse_threshold, postsynapse_threshold, max_distance)
    
    # extract metadata for pixel_size parameter in overlap_um2_coloc
    pixel_size_um, _ , _ = metadata.extract_metadata(file_path)
    print(pixel_size_um)

    # extracting and splitting channels
    pre, post = preprocessing.extract_and_split(file_path, 
                                                presynapse_channel=0, 
                                                postsynapse_channel=1
                                                )
    print(pre, post)
    
    # preprocessing
    p = preprocessing.ImagePreprocessing(
        include_rolling_ball=True, radius=radius,
        include_blur=True, sigma=blur_sigma, preserve_range=True,
        include_clahe=True,
        include_tophat=True, element_size=element_size
        )
    pre_1, post_1 = p.preprocess(pre, post)
    print(pre_1, post_1)
    
    # calculating the coordinates
    presynapse_coord, postsynapse_coord, postsynapse_rot_coord = calc_synaptic_coloc.local_peak_detection(
        presynapse_preprocessed=pre_1,
        postsynapse_preprocessed=post_1,
        presynapse_distance=presynapse_distance,
        postsynapse_distance=postsynapse_distance,
        presynapse_threshold=presynapse_threshold,
        postsynapse_threshold=postsynapse_threshold,
        plot_coord=False
        )
    print(presynapse_coord, postsynapse_coord, postsynapse_rot_coord)
    
    # actual colocalized synapses
    colocalized_spots = calc_synaptic_coloc.count_coloc_spots(
        presynapse_coordinates=presynapse_coord,
        postsynapse_coordinates=postsynapse_coord,
        pixel_size_um=pixel_size_um,
        max_distance_um=max_distance
        )
    print(colocalized_spots)
    
    # rotated colocalized synapses as control
    colocalized_spots_rot = calc_synaptic_coloc.count_coloc_spots(
        presynapse_coordinates=presynapse_coord,
        postsynapse_coordinates=postsynapse_rot_coord,
        pixel_size_um=pixel_size_um,
        max_distance_um=max_distance
        )
    print(colocalized_spots_rot)
    
    # getting the right filename
    print(filename)
    img_filename = metadata.image_filename(filename, name_segments)
    
    return {
        "img_filename": img_filename,
        "local_peak_colocalized_spots": colocalized_spots,
        "local_peak_colocalized_spots_rot": colocalized_spots_rot,
    }

# get a list of files in that input_folder
file_list = os.listdir(input_folder)

# compute the results using a dask delayed object
delayed_results = [local_peak_colocalizaion(filename, name_segments, best_params_df=best_params_by_hippocampal_layer_df, input_folder=input_folder, preprocessing_params=preprocessing_params, synaptic_marker=synaptic_marker) for filename in file_list]
results = dask.compute(*delayed_results)

# reading out the results into a csv
df = pd.DataFrame(results).dropna()  # Drop None results
output_csv_path = os.path.join(output_folder, "local_peak_coloc.csv")
df.to_csv(output_csv_path, encoding="utf-8")


In [ ]:
file = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA3_SL.czi"
pixel_size_um, _ , image_size_um = metadata.extract_metadata(file)
pre, post = preprocessing.extract_and_split(file, presynapse_channel = 0, postsynapse_channel = 1)
p = preprocessing.ImagePreprocessing(
            include_rolling_ball = True, radius = 15, # rolling ball parameters
            include_clahe = False, clip_limit = 0.005, kernel_size = 150, nbins = 265, # CLAHE parameters
            include_tophat = True, element_size = 20, # tophat parameters
            include_blur = True, sigma = 2, preserve_range = True # gaussian blur filters
            )
pre_1, post_1 = p.preprocess(pre, post)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(pre_1, ax=axs[0], label_text = 'VLGUT1')
microshow(post_1, ax=axs[1], label_text = 'PSD95')

In [ ]:
def local_peak_detection(vglut1_preprocessed, psd95_preprocessed, vglut1_distance, psd95_distance, vglut1_threshold, psd95_threshold, plot_coord=False):
    
    """Detects the local intensity peak of each channel processed
    
        Args:
            vglut1_preprocessed (np.array): processed image of vlgut1, which is background substracted and has a gaussian blur
            psd95_preprocessed (np.array): processed image of psd95, which is background substracted and has a gaussian blur
            vglut1_distance (float): minimun distance between vglut1 local peak maxima, usually 1
            psd95_distance (float): mimimun distance between psd95 local peak maxima, usually 1
            vglut1_threshold (float): thresholding of the vlgut1 image for local peak detection
            psd95_threshold (float): thresholding of the psd95 image for local peak detection
    
        Returns:
            vglut1_coord (np.array): coordinates of vglut1 local peak maxima
            psd95_coord (np.array): coordinates of psd95 local peak maxima
            psd95_rot_coord (np.array): coordinates of psd95 local peak maxima
            
            plot of vglut1, psd95, psd95_rot images with the local peak maxima overlaid
    """
    
    # Thresholding the image for local peak maximum detection
    vglut1_coord = peak_local_max(vglut1_preprocessed, min_distance = vglut1_distance, threshold_abs=vglut1_threshold)
    psd95_coord = peak_local_max(psd95_preprocessed, min_distance = psd95_distance, threshold_abs=psd95_threshold)

    # Rotating an image (psd95) as a control
    psd95_rot = rotate(psd95_preprocessed, 90)
    psd95_rot_coord = peak_local_max(psd95_rot, min_distance=psd95_distance, threshold_abs = psd95_threshold)
    
    # Showing the local peaks with coordinates together with the images
    fig, axs = plt.subplots(2, 2, figsize=(30, 30))

    axs[0,0].imshow(vglut1_preprocessed, cmap='gray')
    # axs[0,0].plot(vglut1_coord[:, 1], vglut1_coord[:, 0], 'c.')
    axs[0,0].set_title('vglut1_pre')

    axs[0,1].imshow(psd95_preprocessed, cmap='gray')
    # axs[0,1].plot(psd95_coord[:, 1], psd95_coord[:, 0], 'm.')
    axs[0,1].set_title('psd95_pre')

    axs[1,0].imshow(vglut1_preprocessed, cmap='gray')
    axs[1,0].plot(vglut1_coord[:, 1], vglut1_coord[:, 0], 'c.')
    axs[1,0].set_title('vglut1_pre')

    axs[1,1].imshow(psd95_preprocessed, cmap='gray')
    axs[1,1].plot(psd95_coord[:, 1], psd95_coord[:, 0], 'm.')
    axs[1,1].set_title('psd95_pre')


    # axs[2].imshow(psd95_rot, cmap='gray')
    # axs[2].plot(psd95_rot_coord[:, 1], psd95_rot_coord[:, 0], 'm.')
    # axs[2].set_title('psd95_rot')
    
    return vglut1_coord, psd95_coord, psd95_rot_coord

In [ ]:
# {'pre_distance': 2, 'post_distance': 3, 'pre_threshold': 134.82934781597908, 'post_threshold': 149.97743055799035, 'max_distance_um': 0.025655884409119978}
#  params: 15 20 2 8 7 1428.3721114538846 377.46228118684417 0.35430905603215945
vglut1_coord, psd95_coord, psd95_rot_coord = local_peak_detection(pre_1, post_1, 8, 7, 1428.3721114538846, 377.46228118684417, plot_coord=True)
print(vglut1_coord.shape, psd95_coord.shape, psd95_rot_coord.shape)

In [ ]:

# This block of code calculates the unique colocalized VGLUT1-PSD95 count, meaning it only has unique pairs of colocalized spots.
# It checks of every VLGUT1 and PSD95 spot, whether there are any PSD95 (for VGLUT1) and VLGUT1 (for PSD95) spots in their vicinity within a specific maximum distance and counts it. 
# So it can have for one spot, multiple pairs, and thus including multi-synaptic boutons.

from scipy.spatial.distance import cdist

def count_coloc_spots(vglut1_coordinates, psd95_coordinates, pixel_size_um, max_distance_um):
    
    """Counts the number of colocalized pre and postsynaptic spots based on the coordinates array of the local maxima detection
    
    Args:
        vlgut1_coordinates (np.array): coordinates of the local peak maxima in the vlgut1 channel
        psd95_coordinates (np.array): coordinates of the local peak maxima in the psd95 channel
        pixel_size_um (float): size of a pixel in um, based on image settings
        max_distance_um (float): value of the maximum colocalization distance between the spots of each channel in um
    
    Returns:
        colocalized spot count
    """
    
    # calculate the max distance in pixels with the pixel_size_um and max_distance_um
    max_distance_px = max_distance_um/pixel_size_um
    
    # Calculate pairwise distances between spots in vlgut1 and psd95 channel
    distances_vglut1_to_psd95 = cdist(vglut1_coordinates, psd95_coordinates)
    distances_psd95_to_vglut1 = cdist(psd95_coordinates, vglut1_coordinates)

    # Find unique colocalized spots
    colocalized_spots = set()

    # Iterate over distances from vglut1_coordinates to psd95_coordinates
    for i in range(len(vglut1_coordinates)):
        # Check if the current spot in vlgut1 has nearby spots in psd95
        colocalized_indices = [j for j, distance in enumerate(distances_vglut1_to_psd95[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((i, j))

    # Iterate over distances from psd95_coordinates to vglut1_coordinates
    for i in range(len(psd95_coordinates)):
        # Check if the current spot in Channel 2 has nearby spots in Channel 1
        colocalized_indices = [j for j, distance in enumerate(distances_psd95_to_vglut1[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((j, i))

    # Get the count of unique colocalized spots
    colocalized_spot_count = len(colocalized_spots)
    
    return colocalized_spot_count

In [ ]:
# writing a wrapper function for the two functions, which is easier to implement for the bayesian optimization
def comb_func(vglut1_preprocessed, psd95_preprocessed, vglut1_distance, psd95_distance, vglut1_threshold, psd95_threshold, pixel_size_um, max_distance_um):
    
    # probing the first function to get the parameters as input for the second function
    vglut1_coord, psd95_coord, psd95_rot_coord = local_peak_detection(vglut1_preprocessed, psd95_preprocessed, vglut1_distance, psd95_distance, vglut1_threshold, psd95_threshold)
    
    # probing the second function where psd95 is not rotated, the actual condition
    colocalized_spot_count = count_coloc_spots(vglut1_coord, psd95_coord, pixel_size_um, max_distance_um)
    # probing the second function where psd95 is rotated, the internal control condition
    colocalized_spot_count_rot = count_coloc_spots(vglut1_coord, psd95_rot_coord, pixel_size_um, max_distance_um)
    
    # getting the scale differences between the spot count of the normal situation and psd95 rotated
    scaled_difference = (colocalized_spot_count - colocalized_spot_count_rot) / max(colocalized_spot_count, 1)
    
    return colocalized_spot_count, colocalized_spot_count_rot, scaled_difference

In [ ]:
x, y, z = comb_func(vglut1_preprocessed = pre_1, 
                    psd95_preprocessed = post_1, 
                    vglut1_distance = 8, 
                    psd95_distance = 7, 
                    vglut1_threshold = 1428.3721114538846, 
                    psd95_threshold = 377.46228118684417 , 
                    pixel_size_um = pixel_size_um, 
                    max_distance_um = 0.35430905603215945)
print(x, y, z)

In [ ]:
import optuna

# setting the static variables
vglut1_preprocessed = pre_1 # Assign vglut1_pre to vglut1_preprocessed
psd95_preprocessed = post_1  # Assign psd95_pre to psd95_preprocessed
pixel_size_um = pixel_size_um  # Assign pixel_size to pixel_size_um


def objective(trial):
    
    vglut1_distance = trial.suggest_int("vglut1_distance", 1, 3)
    psd95_distance = trial.suggest_int("psd95_distance", 1, 3)
    vglut1_threshold = trial.suggest_float("vlgut1_threshold", 100, 200)
    psd95_threshold = trial.suggest_float("psd95_threshold", 50, 150)
    max_distance_um = trial.suggest_float("max_distance_um", 0.01, 1)

    _, _, scaled_spot_count_dif = comb_func(
        vglut1_preprocessed = vglut1_preprocessed,
        psd95_preprocessed = psd95_preprocessed,        
        vglut1_distance =vglut1_distance, 
        psd95_distance = psd95_distance,
        vglut1_threshold = vglut1_threshold,
        psd95_threshold = psd95_threshold,
        pixel_size_um = pixel_size_um,
        max_distance_um = max_distance_um
    )

    return scaled_spot_count_dif 


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=500)
print(study.best_trial.value)

In [ ]:
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline



plot_rank(study)